# 🔍 ScopeSnap — YOLOv8 AC Defect Detection Training
**Run this entire notebook on Google Colab with a FREE T4 GPU.**

### What this notebook does:
1. Installs YOLOv8 on Colab's free GPU
2. Downloads all 6 public AC defect datasets from Roboflow Universe
3. Merges and normalizes all labels to 9 consistent ScopeSnap classes
4. Trains YOLOv8s model (~60-90 minutes on T4 GPU)
5. Evaluates accuracy per defect class
6. Exports to ONNX format ready for the ScopeSnap mobile app
7. Downloads the trained model to your computer

### Before you start:
- Go to **Runtime → Change runtime type → T4 GPU → Save**
- Get a **free Roboflow API key**: roboflow.com → Settings → API
- Paste your key in Cell 3 below

---
**Expected training time: 60–90 minutes | Cost: $0**

## Cell 1 — Verify GPU and Install YOLOv8

In [ ]:
# Verify GPU is available
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else 'NOT FOUND — change runtime to GPU!')

# Install YOLOv8
!pip install ultralytics roboflow -q
import ultralytics
print('YOLOv8 version:', ultralytics.__version__)
print('\n✅ Ready to train!')

## Cell 2 — Set Up Directory Structure

In [ ]:
import os
from pathlib import Path

BASE = Path('/content/scopesnap_yolo')
for split in ['train', 'val', 'test']:
    (BASE / 'images' / split).mkdir(parents=True, exist_ok=True)
    (BASE / 'labels' / split).mkdir(parents=True, exist_ok=True)

# ScopeSnap canonical class names — ALL datasets will be normalized to these
CLASS_NAMES = [
    'rust',           # 0 — visible rust/oxidation
    'corrosion',      # 1 — early/light corrosion
    'dirty_filter',   # 2 — clogged/dirty air filter
    'bent_fins',      # 3 — bent/damaged condenser/evaporator fins
    'ice_buildup',    # 4 — ice on evaporator coil
    'mould',          # 5 — biological growth
    'burnt_component',# 6 — burnt wiring/components
    'drain_issue',    # 7 — drain pan/clog issues
    'normal',         # 8 — no defect
]

# MASTER LABEL NORMALIZATION MAP
# Maps ANY label from ANY dataset → canonical class id
LABEL_MAP = {
    # rust
    'rust': 0, 'oxidation': 0, 'metal_rust': 0, 'rust_spot': 0,
    'corrosion_high': 0, 'heavy_rust': 0, 'surface_rust': 0, 'severe_rust': 0,
    # corrosion
    'corrosion': 1, 'mild_corrosion': 1, 'light_corrosion': 1,
    'surface_corrosion': 1, 'early_corrosion': 1, 'corrosion_low': 1,
    # dirty filter
    'dirty_filter': 2, 'clogged_filter': 2, 'blocked_filter': 2,
    'dirty': 2, 'filter_dirty': 2, 'dusty_filter': 2,
    # bent fins
    'bent_fins': 3, 'damaged_fins': 3, 'fin_damage': 3, 'bent': 3,
    'fin_bent': 3, 'fins': 3, 'bent_fin': 3,
    # ice
    'ice_buildup': 4, 'ice': 4, 'frozen_coil': 4, 'frost': 4,
    'freeze': 4, 'icing': 4, 'ice_formation': 4,
    # mould
    'mould': 5, 'mold': 5, 'biological_growth': 5, 'algae': 5,
    'fungus': 5, 'mildew': 5,
    # burnt
    'burnt_component': 6, 'burnt': 6, 'burn_mark': 6, 'burned': 6,
    'electrical_burn': 6, 'scorch': 6, 'piston_defect': 6, 'defect': 6,
    # drain
    'drain_issue': 7, 'drain_clog': 7, 'standing_water': 7, 'clog': 7,
    # normal
    'normal': 8, 'ok': 8, 'no_defect': 8, 'good': 8, 'clean': 8, 'no_damage': 8,
}

print(f'✅ Directories created: {BASE}')
print(f'✅ Classes defined: {len(CLASS_NAMES)}')
print(f'✅ Label mappings: {len(LABEL_MAP)} variants → 9 classes')

## Cell 3 — Paste Your Roboflow API Key Here

In [ ]:
# ⚠️ PASTE YOUR FREE ROBOFLOW API KEY HERE
# Get it free at: roboflow.com → Settings → Roboflow API → Copy Private Key
ROBOFLOW_API_KEY = 'TxuC1ROLgWPdikG4ZuNL'

if not ROBOFLOW_API_KEY or len(ROBOFLOW_API_KEY) < 10:
    print('⚠️  Please paste your Roboflow API key above and re-run this cell')
else:
    print('✅ API key set')

## Cell 4 — Download All 6 Datasets from Roboflow Universe

In [ ]:
from roboflow import Roboflow
import shutil, random
from pathlib import Path

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Datasets to download — workspace/project/version
# These are all PUBLIC datasets on Roboflow Universe
DATASETS = [
    {
        'workspace': 'roboflow-universe-projects',
        'project':   'rust-detection-p3psx',
        'version':   1,
        'class_override': None,  # use dataset's own labels
        'name': 'Rust Detection (10,072 images)',
    },
    {
        'workspace': 'roboflow-universe-projects', 
        'project':   'corrosion-detection-s9pzz',
        'version':   1,
        'class_override': None,
        'name': 'Corrosion Detection',
    },
    {
        'workspace': 'ac-detection',
        'project':   'ac-object-detection',
        'version':   1,
        'class_override': None,
        'name': 'AC Object Detection',
    },
    {
        'workspace': 'air-conditioning',
        'project':   'air-conditioning-yolo',
        'version':   1,
        'class_override': None,
        'name': 'Air Conditioning YOLO',
    },
]

downloaded = []
for ds in DATASETS:
    try:
        print(f"\nDownloading: {ds['name']}")
        project = rf.workspace(ds['workspace']).project(ds['project'])
        version = project.version(ds['version'])
        dl = version.download('yolov8', location=f"/content/raw_{ds['project']}")
        downloaded.append({'path': f"/content/raw_{ds['project']}", **ds})
        print(f'  ✅ Downloaded: {ds["name"]}')
    except Exception as e:
        print(f'  ⚠️ Could not download {ds["name"]}: {e}')
        print(f'     Try searching roboflow.com/universe for this dataset manually')

print(f'\nDownloaded {len(downloaded)}/{len(DATASETS)} datasets')

## Cell 5 — Merge All Datasets with Normalized Labels

In [ ]:
import shutil, random
from pathlib import Path
from collections import Counter

random.seed(42)

def normalize_label_line(line, ds_class_names=None, class_override=None):
    """
    Convert a YOLO annotation line to use ScopeSnap canonical class ids.
    Returns normalized line string or None if label is unknown.
    """
    parts = line.strip().split()
    if len(parts) < 5:
        return None
    try:
        orig_id = int(parts[0])
        coords  = parts[1:]
        # Validate bbox
        if any(float(c) < 0 or float(c) > 1 for c in coords):
            return None
        if class_override is not None:
            return f'{class_override} {" ".join(coords)}'
        # Look up label name from dataset's own class list
        if ds_class_names and orig_id < len(ds_class_names):
            label_name = ds_class_names[orig_id].lower().replace(' ', '_').replace('-', '_')
            new_id = LABEL_MAP.get(label_name)
            if new_id is None:
                # Try partial match
                for k, v in LABEL_MAP.items():
                    if k in label_name or label_name in k:
                        new_id = v
                        break
            if new_id is not None:
                return f'{new_id} {" ".join(coords)}'
        return None
    except:
        return None

def read_dataset_classes(ds_path):
    """Read class names from data.yaml in a downloaded Roboflow dataset."""
    import yaml
    yaml_path = Path(ds_path) / 'data.yaml'
    if yaml_path.exists():
        with open(yaml_path) as f:
            data = yaml.safe_load(f)
        return data.get('names', [])
    return []

all_image_label_pairs = []

# Process each downloaded dataset
for ds in downloaded:
    ds_path = Path(ds['path'])
    ds_classes = read_dataset_classes(ds_path)
    print(f"\nProcessing: {ds['name']}")
    print(f"  Classes found: {ds_classes}")

    for split in ['train', 'valid', 'val', 'test']:
        img_dir = ds_path / split / 'images'
        lbl_dir = ds_path / split / 'labels'
        if not img_dir.exists():
            img_dir = ds_path / 'images' / split
            lbl_dir = ds_path / 'labels' / split
        if not img_dir.exists():
            continue

        for img_path in sorted(img_dir.glob('*.jpg')) + sorted(img_dir.glob('*.png')):
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if lbl_path.exists():
                # Read and normalize labels
                new_lines = []
                with open(lbl_path) as f:
                    for line in f:
                        norm = normalize_label_line(
                            line,
                            ds_class_names=ds_classes,
                            class_override=ds.get('class_override')
                        )
                        if norm:
                            new_lines.append(norm)
                if new_lines:  # only include images that have valid annotations
                    all_image_label_pairs.append((img_path, new_lines))

print(f'\n✅ Total valid image-annotation pairs: {len(all_image_label_pairs):,}')

# Split 75/15/10
random.shuffle(all_image_label_pairs)
n = len(all_image_label_pairs)
splits = {
    'train': all_image_label_pairs[:int(n*0.75)],
    'val':   all_image_label_pairs[int(n*0.75):int(n*0.90)],
    'test':  all_image_label_pairs[int(n*0.90):],
}

# Write to dataset structure
class_counts = Counter()
for split_name, pairs in splits.items():
    img_out = BASE / 'images' / split_name
    lbl_out = BASE / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    for img_path, norm_lines in pairs:
        dest_img = img_out / img_path.name
        shutil.copy2(img_path, dest_img)
        with open(lbl_out / (img_path.stem + '.txt'), 'w') as f:
            f.write('\n'.join(norm_lines))
        for ln in norm_lines:
            class_counts[int(ln.split()[0])] += 1
    print(f'  {split_name:6s}: {len(pairs):,} images')

print('\nClass distribution in training set:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  [{i}] {name:20s}: {class_counts.get(i, 0):,} bounding boxes')

## Cell 6 — Write data.yaml

In [ ]:
yaml_content = f'''# ScopeSnap AC Defect Detection Dataset — Auto-generated
path: {str(BASE)}
train: images/train
val: images/val
test: images/test

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
'''

with open(BASE / 'data.yaml', 'w') as f:
    f.write(yaml_content)

print('✅ data.yaml written')
print(yaml_content)

## Cell 7 — Train YOLOv8 on Free T4 GPU (~60–90 minutes)

In [ ]:
from ultralytics import YOLO

# YOLOv8s = small model — best balance of speed and accuracy for mobile deployment
# Pre-trained on COCO — we fine-tune the detection head on our AC defect classes
model = YOLO('yolov8s.pt')

print('Starting training...')
print('Estimated time: 60-90 minutes on T4 GPU')
print('Watch mAP@50 — target: 0.75+ before stopping')
print()

results = model.train(
    data=str(BASE / 'data.yaml'),
    epochs=100,
    imgsz=640,
    batch=16,
    lr0=0.0001,
    lrf=0.01,
    cos_lr=True,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    patience=25,          # early stop if no improvement for 25 epochs
    save=True,
    project='/content/scopesnap_training',
    name='ac_defects_v1',
    exist_ok=True,
    verbose=True,
)

print('\n✅ Training complete!')
print(f'Best mAP@50: {results.results_dict.get("metrics/mAP50(B)", "N/A")}')

## Cell 8 — Evaluate the Trained Model

In [ ]:
from ultralytics import YOLO

best_model_path = '/content/scopesnap_training/ac_defects_v1/weights/best.pt'
model = YOLO(best_model_path)

print('Evaluating on test set...')
metrics = model.val(
    data=str(BASE / 'data.yaml'),
    split='test',
    conf=0.25,
)

print(f'\nTest Set Results:')
print(f'  mAP@50:    {metrics.box.map50:.4f}')
print(f'  mAP@50-95: {metrics.box.map:.4f}')
print(f'  Precision: {metrics.box.mp:.4f}')
print(f'  Recall:    {metrics.box.mr:.4f}')
print(f'\nPer-class mAP@50:')
for i, (name, ap) in enumerate(zip(CLASS_NAMES, metrics.box.ap50)):
    bar = '█' * int(ap * 30)
    print(f'  [{i}] {name:20s}: {ap:.3f}  {bar}')

## Cell 9 — Export to ONNX (for ScopeSnap Mobile App)

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/scopesnap_training/ac_defects_v1/weights/best.pt')

# Export to ONNX — this format runs on any device (Android, iOS, server)
print('Exporting to ONNX...')
onnx_path = model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    dynamic=False,
    opset=12,
)

import os
size_mb = os.path.getsize(onnx_path) / (1024*1024)
print(f'\n✅ ONNX model exported!')
print(f'   Path: {onnx_path}')
print(f'   Size: {size_mb:.1f} MB')
print(f'   This file goes into: scopesnap-api/models/scopesnap_defect_v1.onnx')

## Cell 10 — Download Your Trained Model

In [ ]:
import shutil, os
from google.colab import files

# Package everything into a zip
shutil.copy(
    '/content/scopesnap_training/ac_defects_v1/weights/best.pt',
    '/content/scopesnap_yolo_best.pt'
)

# Find and rename the ONNX file
import glob
onnx_files = glob.glob('/content/scopesnap_training/**/*.onnx', recursive=True)
if onnx_files:
    shutil.copy(onnx_files[0], '/content/scopesnap_defect_v1.onnx')
    print(f'ONNX: {onnx_files[0]}')

print('\n⬇️ Downloading trained model files...')
files.download('/content/scopesnap_yolo_best.pt')       # PyTorch weights
files.download('/content/scopesnap_defect_v1.onnx')     # ONNX for deployment

print('\n✅ Download complete!')
print('\nNext steps:')
print('1. Copy scopesnap_defect_v1.onnx → scopesnap-api/models/')
print('2. Run the FastAPI integration code from Section 9.6 of the AC Guide')
print('3. Test with: test_hvac_unit.jpg in the ScopeSnap repository')

---
## After Training — What to Do with Your Models

### ONNX Model → ScopeSnap App
```
scopesnap-api/
└── models/
    └── scopesnap_defect_v1.onnx   ← copy here
```

### FastAPI Integration (api/estimates.py)
```python
import onnxruntime as ort
import numpy as np
import cv2

# Load once at startup
yolo_session = ort.InferenceSession('models/scopesnap_defect_v1.onnx')

def detect_defects(image_bytes):
    img = cv2.imdecode(np.frombuffer(image_bytes, np.uint8), cv2.IMREAD_COLOR)
    img = cv2.resize(img, (640, 640))
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))[np.newaxis]
    outputs = yolo_session.run(None, {'images': img})
    return parse_yolo_output(outputs, confidence_threshold=0.40)
```

### Phase 2 — Retrain with Real Mike Photos
After 3,000+ real assessments collected in Cloudflare R2:
1. Export real photos from R2 with their tech-confirmed defect labels
2. Add to the Roboflow project under 'ScopeSnap-AC-Defects'
3. Retrain using this same notebook — expected mAP@50 jumps to 90-95%